In [15]:
from dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain.tools import tool
from langchain.agents import create_agent

In [2]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitter_data = splitter.split_documents(docs)
len(splitter_data)

26

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_stroe = Chroma.from_documents(
    documents=splitter_data,
    embedding=embeddings 
)

#### Agent build: we need - tools, llm, prompt

In [33]:
@tool
def retrieve_tool(query:str):
    """
    This tool can help to retrieve the relevant data of the PDF Documents, and these PDF documents have details about the medical report of a patient. 
    """
    print("Tool Called:", query)
    data = vector_stroe.similarity_search(query=query, k=4)
    context = ""
    for doc in data:
        context += doc.page_content + "\n\n"

    return context    

In [34]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [36]:
System_prompt = """
    You are a helpful assistant that answer question using retrived context.
    Always use the 'retriever_tool' tool for question requiring external knowledge.
"""

In [37]:
agent = create_agent(
    model=llm,
    tools=[retrieve_tool],
    system_prompt=System_prompt
)

In [39]:
query = "What is the name of Patient? and what are the names of the Doctors?"
response = agent.invoke({"messages":[{"role":"user","content":query}]})
result = response["messages"][-1].content
print(result[0]["text"])

Tool Called: patient name doctor name medical report
Based on the medical report:

* **Patient Name:** Ms. Nikita Chudhary
* **Doctors / Referring & Consulting Physicians:**
  * **Dr. Nitin Nahar** (Referring Doctor / Ref By)
  * **Dr. Sunanda** (Sr. Consultant Pathologist - Hematology & Immunology)


In [ ]:
from dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain.tools import tool
from langchain.agents import create_agent

loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitter_data = splitter.split_documents(docs)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_stroe = Chroma.from_documents(
    documents=splitter_data,
    embedding=embeddings 
)
@tool
def retrieve_tool(query:str):
    """
    This tool can help to retrieve the relevant data of the PDF Documents, 
    and these PDF documents have details about the medical report of a patient. 
    """
    print("Tool Called:", query)
    data = vector_stroe.similarity_search(query=query, k=4)
    context = ""
    for doc in data:
        context += doc.page_content + "\n\n"

    return context

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
System_prompt = """
    You are a helpful assistant that answer question using retrived context.
    Always use the 'retriever_tool' tool for question requiring external knowledge.
"""
agent = create_agent(
    model=llm,
    tools=[retrieve_tool],
    system_prompt=System_prompt
)
query = "What is the name of Patient? and what are the names of the Doctors?"
response = agent.invoke({"messages":[{"role":"user","content":query}]})
result = response["messages"][-1].content
print(result[0]["text"])